# MedFlow — 01 · Engenharia de dados (camada Silver)

Este notebook transforma a Bronze em bases analíticas rastreáveis. Aqui ficam
todos os tratamentos, de/paras, dimensões, fatos, flags de qualidade e
reconciliações.

## Os oito controles desta revisão

1. inventariar campos e domínios;
2. completar os de/paras conhecidos;
3. preservar `N_AIH`, `IDENT` e `COD_IDADE`;
4. separar AIH aprovada de internação nova;
5. separar `QT_DIARIAS` de `DIAS_PERM`;
6. classificar região ausente/conflitante sem inventar domínio;
7. impedir perdas por `groupby` com nulos;
8. revalidar os insumos dos indicadores e bloquear interpretações não sustentadas.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import calendar
import json
import sys
from zipfile import ZipFile
import numpy as np
import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 80)

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_BRONZE = BASE / "dados" / "bronze"
DIR_BRONZE_PARQUET = DIR_BRONZE / "parquet"
DIR_REFERENCIAS = DIR_BRONZE / "origem" / "referencias"
SOBRESCREVER = False

manifesto = json.loads((DIR_BRONZE / "MANIFESTO.json").read_text(encoding="utf-8"))
assert manifesto["camada"] == "bronze"
COMPETENCIAS = [
    (int(valor[:4]), int(valor[4:]))
    for valor in manifesto["recorte"]["competencias"]
]
ARQ_SIH = DIR_BRONZE_PARQUET / manifesto["arquivos"]["sih"]["caminho"]
ARQ_CNES = DIR_BRONZE_PARQUET / manifesto["arquivos"]["cnes"]["caminho"]
print("entrada:", DIR_BRONZE_PARQUET.relative_to(BASE))
print("saída  :", (BASE / "dados" / "silver").relative_to(BASE))

entrada: dados/bronze/parquet
saída  : dados/silver


## 1. Carga e inventário

O inventário mede a cobertura dos códigos observados e registra a proveniência
de cada de/para. Código desconhecido nunca vira categoria genérica silenciosamente.

In [2]:
COLS_SIH = [
    "N_AIH", "IDENT", "CNES", "MUNIC_MOV", "MUNIC_RES", "ESPEC",
    "QT_DIARIAS", "DIAS_PERM", "MORTE", "VAL_TOT", "DIAG_PRINC",
    "DT_INTER", "DT_SAIDA", "IDADE", "COD_IDADE", "SEXO", "CAR_INT",
    "COMPLEX", "MARCA_UTI", "UTI_MES_TO", "ANO_CMPT", "MES_CMPT",
    "_arquivo_fonte", "_ano_arquivo", "_mes_arquivo",
]
sih = pd.read_parquet(ARQ_SIH, columns=COLS_SIH)
cnes = pd.read_parquet(ARQ_CNES)
ibge_raw = json.loads((DIR_REFERENCIAS / "ibge_municipios_sp_raw.json").read_bytes())
regioes_ms_payload = json.loads((DIR_REFERENCIAS / "ms_regioes_saude_sp_raw.json").read_bytes())
cnes_atual_payload = json.loads((DIR_REFERENCIAS / "ms_cnes_estabelecimentos_atuais_raw.json").read_bytes())
ARQ_CID10 = DIR_REFERENCIAS / "datasus_cid10_2008.zip"
ARQ_REGIOES_CSV = DIR_REFERENCIAS / "macrorregiao_de_saude_csv.zip"
with ZipFile(ARQ_REGIOES_CSV) as pacote_regioes:
    with pacote_regioes.open("macroregiao_de_saude.csv") as arquivo_regioes:
        regioes_csv = pd.read_csv(arquivo_regioes, sep=";", dtype="string")
regioes_csv_sp = regioes_csv[regioes_csv.sg_uf.eq("SP")].copy()
regioes_csv_sp["municipio_cod6"] = regioes_csv_sp.cod_municipio.str.zfill(6)
regioes_csv_sp["qt_populacao_ibge_2022"] = pd.to_numeric(
    regioes_csv_sp.populacao_ibge_2022, errors="raise"
).astype("int64")

for coluna in ["N_AIH", "IDENT", "CNES", "MUNIC_MOV", "MUNIC_RES", "ESPEC",
               "DIAG_PRINC", "COD_IDADE", "SEXO", "CAR_INT", "COMPLEX",
               "MARCA_UTI", "DT_INTER", "DT_SAIDA"]:
    sih[coluna] = sih[coluna].astype("string").str.strip()
for coluna in ["QT_DIARIAS", "DIAS_PERM", "MORTE", "IDADE", "UTI_MES_TO"]:
    sih[coluna] = pd.to_numeric(sih[coluna], errors="raise").astype("int32")
sih["VAL_TOT"] = pd.to_numeric(sih["VAL_TOT"], errors="raise").astype("float64")
sih["_ano"] = pd.to_numeric(sih["ANO_CMPT"], errors="raise").astype("int16")
sih["_mes"] = pd.to_numeric(sih["MES_CMPT"], errors="raise").astype("int8")

print(f"SIH : {sih.shape[0]:,} × {sih.shape[1]}")
print(f"CNES: {cnes.shape[0]:,} × {cnes.shape[1]}")
print("IDENT:", sih.IDENT.value_counts(dropna=False).to_dict())

SIH : 7,034,961 × 27
CNES: 243,085 × 31
IDENT: {'1': 6905441, '5': 129520}


In [3]:
DEPARA_ESPEC = {
    "01": "Cirurgia", "02": "Obstetrícia", "03": "Clínica médica",
    "04": "Crônicos", "05": "Psiquiatria", "06": "Tisiologia",
    "07": "Pediatria", "08": "Reabilitação",
    "09": "Hospital-dia (cirúrgico)", "10": "Aids - hospital-dia",
    "11": "Fibrose cística - hospital-dia",
    "12": "Intercorrência pós-transplante - hospital-dia",
    "13": "Geriatria - hospital-dia", "14": "Saúde mental - hospital-dia",
    "17": "Estabelecimento exclusivo UTI SUS",
    "87": "Saúde mental - clínico",
}
DEPARA_IDENT = {"1": "AIH normal / internação nova", "5": "AIH de continuação / longa permanência"}
DEPARA_COD_IDADE = {
    "0": "Ignorado", "2": "Dias", "3": "Meses", "4": "Anos",
    "5": "Anos acima de 100 (valor adicional)",
}
DEPARA_SEXO = {"0": "Ignorado", "1": "Masculino", "3": "Feminino"}
DEPARA_CAR_INT = {
    "01": "Eletivo", "02": "Urgência",
    "03": "Acidente no local de trabalho ou a serviço",
    "04": "Acidente no trajeto para o trabalho",
    "05": "Outros acidentes de trânsito",
    "06": "Outros tipos de lesões e envenenamentos",
}
DEPARA_COMPLEX = {"02": "Média complexidade", "03": "Alta complexidade"}
DEPARA_TP_UNID = {
    "05": "Hospital Geral", "07": "Hospital Especializado", "15": "Unidade Mista",
    "20": "Pronto Socorro Geral", "21": "Pronto Socorro Especializado",
    "36": "Clínica/Centro de Especialidade", "62": "Hospital/Dia Isolado",
    "73": "Pronto Atendimento",
}
DEPARA_GESTAO = {"M": "Municipal", "E": "Estadual"}
DEPARA_NAT_JUR = {
    "1015": "Órgão Público do Poder Executivo Federal",
    "1023": "Órgão Público do Poder Executivo Estadual ou do Distrito Federal",
    "1031": "Órgão Público do Poder Executivo Municipal",
    "1104": "Autarquia Federal", "1112": "Autarquia Estadual ou do Distrito Federal",
    "1120": "Autarquia Municipal",
    "1155": "Fundação Pública de Direito Público Municipal",
    "1210": "Consórcio Público de Direito Público (Associação Pública)",
    "1228": "Consórcio Público de Direito Privado",
    "1244": "Município", "1279": "Fundação Pública de Direito Privado Municipal",
    "2011": "Empresa Pública", "2046": "Sociedade Anônima Aberta",
    "2054": "Sociedade Anônima Fechada", "2062": "Sociedade Empresária Limitada",
    "2135": "Empresário (Individual)", "2143": "Cooperativa",
    "2232": "Sociedade Simples Pura", "2240": "Sociedade Simples Limitada",
    "2305": "Empresa Individual de Responsabilidade Limitada (Natureza Empresária)",
    "2313": "Empresa Individual de Responsabilidade Limitada (Natureza Simples)",
    "3069": "Fundação Privada", "3999": "Associação Privada",
}
DEPARA_UTI = {
    "00": "Sem marca de UTI", "01": "UTI Adulto nível II (código legado)",
    "51": "UTI II Adulto COVID-19", "52": "UTI II Pediátrica COVID-19",
    "74": "UTI Adulto I", "75": "UTI Adulto II", "76": "UTI Adulto III",
    "77": "UTI Pediátrica I", "78": "UTI Pediátrica II", "79": "UTI Pediátrica III",
    "80": "UTI Neonatal I", "81": "UTI Neonatal II", "82": "UTI Neonatal III",
    "83": "UTI de queimados", "85": "UCO II", "86": "UCO III",
    "99": "Não utilizou UTI",
}

dominios = [
    ("ESPEC", DEPARA_ESPEC, "DATASUS/TabNet", "mapeado"),
    ("IDENT", DEPARA_IDENT, "DATASUS/TabNet", "mapeado"),
    ("COD_IDADE", DEPARA_COD_IDADE, "Dicionário SIH", "mapeado"),
    ("SEXO", DEPARA_SEXO, "Dicionário SIH", "mapeado"),
    ("CAR_INT", DEPARA_CAR_INT, "Dicionário SIH", "mapeado"),
    ("COMPLEX", DEPARA_COMPLEX, "Dicionário SIH", "mapeado"),
    ("TP_UNID", DEPARA_TP_UNID, "CNES", "mapeado"),
    ("TPGESTAO", DEPARA_GESTAO, "CNES", "mapeado"),
    ("MARCA_UTI", DEPARA_UTI, "MS/DATASUS + CEM", "mapeado_multifonte"),
    ("NAT_JUR", DEPARA_NAT_JUR, "CONCLA/IBGE 2021", "mapeado"),
]
dim_dominio = pd.DataFrame([
    {"campo": campo, "codigo": codigo, "descricao": descricao,
     "fonte": fonte, "status": status}
    for campo, mapa, fonte, status in dominios
    for codigo, descricao in mapa.items()
])

inventario = []
for campo, mapa, fonte, status in dominios[:6] + [dominios[8]]:
    observados = set(sih[campo].dropna().astype(str).unique())
    inventario.append({
        "campo": campo, "qtd_codigos_observados": len(observados),
        "codigos_sem_depara": ", ".join(sorted(observados - set(mapa))) or "nenhum",
        "cobertura_linhas_pct": round(sih[campo].isin(mapa).mean() * 100, 6),
        "fonte": fonte, "status": status,
    })
print(pd.DataFrame(inventario).to_string(index=False))
assert sih.ESPEC.isin(DEPARA_ESPEC).all(), "há especialidade sem de/para"
assert sih.IDENT.isin(DEPARA_IDENT).all(), "há IDENT não classificado"

    campo  qtd_codigos_observados codigos_sem_depara  cobertura_linhas_pct            fonte             status
    ESPEC                      15             nenhum                 100.0   DATASUS/TabNet            mapeado
    IDENT                       2             nenhum                 100.0   DATASUS/TabNet            mapeado
COD_IDADE                       5             nenhum                 100.0   Dicionário SIH            mapeado
     SEXO                       2             nenhum                 100.0   Dicionário SIH            mapeado
  CAR_INT                       6             nenhum                 100.0   Dicionário SIH            mapeado
  COMPLEX                       2             nenhum                 100.0   Dicionário SIH            mapeado
MARCA_UTI                      14             nenhum                 100.0 MS/DATASUS + CEM mapeado_multifonte


## 2. Dimensões

Região de saúde usa a declaração do próprio estabelecimento. A inferência pelo
município só ocorre quando existe exatamente um código válido naquele
município. Conflito, inferência e ausência ficam explícitos.

In [4]:
dim_tempo = pd.DataFrame(COMPETENCIAS, columns=["_ano", "_mes"])
dim_tempo["dias_no_mes"] = [calendar.monthrange(a, m)[1] for a, m in zip(dim_tempo._ano, dim_tempo._mes)]
dim_tempo["competencia"] = dim_tempo._ano.astype(str) + dim_tempo._mes.astype(str).str.zfill(2)
dim_tempo["data_ref"] = pd.to_datetime(dim_tempo.competencia + "01", format="%Y%m%d")
dim_tempo["trimestre"] = dim_tempo.data_ref.dt.quarter

obs_espec = sih.ESPEC.value_counts().rename_axis("ESPEC").reset_index(name="aih_aprovadas")
dim_especialidade = obs_espec.assign(
    especialidade=obs_espec.ESPEC.map(DEPARA_ESPEC),
    mapeada=obs_espec.ESPEC.isin(DEPARA_ESPEC).astype("int8"),
)

CAPITULOS_CID = [
    ("I","A00","B99","Infecciosas e parasitárias"), ("II","C00","D48","Neoplasias"),
    ("III","D50","D89","Sangue e órgãos hematopoéticos"), ("IV","E00","E90","Endócrinas, nutricionais e metabólicas"),
    ("V","F00","F99","Transtornos mentais e comportamentais"), ("VI","G00","G99","Sistema nervoso"),
    ("VII","H00","H59","Olho e anexos"), ("VIII","H60","H95","Ouvido e apófise mastoide"),
    ("IX","I00","I99","Aparelho circulatório"), ("X","J00","J99","Aparelho respiratório"),
    ("XI","K00","K93","Aparelho digestivo"), ("XII","L00","L99","Pele e tecido subcutâneo"),
    ("XIII","M00","M99","Osteomuscular e tecido conjuntivo"), ("XIV","N00","N99","Aparelho geniturinário"),
    ("XV","O00","O99","Gravidez, parto e puerpério"), ("XVI","P00","P96","Afecções do período perinatal"),
    ("XVII","Q00","Q99","Malformações congênitas"), ("XVIII","R00","R99","Sintomas e achados anormais"),
    ("XIX","S00","T98","Lesões e envenenamentos"), ("XX","V01","Y98","Causas externas"),
    ("XXI","Z00","Z99","Fatores que influenciam o estado de saúde"), ("XXII","U04","U99","Propósitos especiais"),
]
def classificar_cid(valor):
    chave = str(valor).upper()[:3]
    for capitulo, inicio, fim, descricao in CAPITULOS_CID:
        if inicio <= chave <= fim:
            return capitulo, descricao
    return "--", "Não classificado"

with ZipFile(ARQ_CID10) as pacote:
    cid_sub = pd.read_csv(
        pacote.open("CID-10-SUBCATEGORIAS.CSV"), sep=";", encoding="latin1", dtype=str
    )
    cid_cat = pd.read_csv(
        pacote.open("CID-10-CATEGORIAS.CSV"), sep=";", encoding="latin1", dtype=str
    )
cid_sub = cid_sub.rename(columns={
    "SUBCAT": "cid_principal", "DESCRICAO": "cid_descricao",
    "DESCRABREV": "cid_descricao_abreviada",
})[["cid_principal", "cid_descricao", "cid_descricao_abreviada"]]
cid_sub["fonte_descricao"] = "DATASUS CID-10 2008 — subcategoria"
cid_cat_ref = cid_cat.rename(columns={
    "CAT": "cid_principal", "DESCRICAO": "cid_descricao",
    "DESCRABREV": "cid_descricao_abreviada",
})[["cid_principal", "cid_descricao", "cid_descricao_abreviada"]]
cid_cat_ref["fonte_descricao"] = "DATASUS CID-10 2008 — categoria"
cid_ref = pd.concat([cid_sub, cid_cat_ref], ignore_index=True).drop_duplicates("cid_principal")

CID_COMPLEMENTAR = {
    "U09": ("Condição pós-COVID-19", "Ministério da Saúde — condição pós-COVID"),
    "U099": ("Condição de saúde posterior à COVID-19, não especificada", "Ministério da Saúde — condição pós-COVID"),
    "U10": ("Síndrome inflamatória multissistêmica associada à COVID-19", "Ministério da Saúde — orientação COVID-19"),
    "U109": ("Síndrome inflamatória multissistêmica associada à COVID-19, não especificada", "Ministério da Saúde — orientação COVID-19"),
    "N182": ("Doença renal crônica, estágio 2", "Ministério da Saúde — PCDT DRC 2024"),
    "N183": ("Doença renal crônica, estágio 3", "Ministério da Saúde — PCDT DRC 2024"),
    "N184": ("Doença renal crônica, estágio 4", "Ministério da Saúde — PCDT DRC 2024"),
    "N185": ("Doença renal crônica, estágio 5", "Ministério da Saúde — PCDT DRC 2024"),
    "C824": ("Linfoma folicular grau IIIb", "Ministério da Saúde — RTS/SIGTAP, Portaria SAES 2.203/2024"),
    "C826": ("Linfoma cutâneo do centro do folículo", "Ministério da Saúde — RTS/SIGTAP, Portaria SAES 2.203/2024"),
}
cid_complementar = pd.DataFrame([
    {"cid_principal": codigo, "cid_descricao": descricao,
     "cid_descricao_abreviada": descricao, "fonte_descricao": fonte}
    for codigo, (descricao, fonte) in CID_COMPLEMENTAR.items()
])
cid_ref = pd.concat([cid_ref, cid_complementar], ignore_index=True).drop_duplicates(
    "cid_principal", keep="last"
)

codigos = sih.DIAG_PRINC.value_counts(dropna=False).rename_axis("cid_principal").reset_index(name="aih_aprovadas")
dim_cid = codigos.merge(cid_ref, on="cid_principal", how="left", validate="one_to_one")
dim_cid["categoria"] = dim_cid.cid_principal.str[:3]
categorias = cid_cat[["CAT", "DESCRICAO"]].rename(
    columns={"CAT": "categoria", "DESCRICAO": "categoria_descricao"}
).drop_duplicates("categoria")
dim_cid = dim_cid.merge(categorias, on="categoria", how="left", validate="many_to_one")
classificacao = dim_cid.cid_principal.map(classificar_cid)
dim_cid["capitulo"] = [x[0] for x in classificacao]
dim_cid["capitulo_desc"] = [x[1] for x in classificacao]
assert (dim_cid.capitulo != "--").all(), "há CID sem capítulo"
assert dim_cid.cid_descricao.notna().all(), "há CID sem descrição oficial/confiável"


In [5]:
def normaliza_codigo(valor, largura=None):
    texto = str(valor).strip()
    if not texto or texto.lower() in {"nan", "none"} or not texto.isdigit():
        return pd.NA
    return texto.zfill(largura) if largura else texto

for coluna in ["CNES", "CODUFMUN", "TP_UNID", "ESFERA_A", "NAT_JUR", "TPGESTAO"]:
    cnes[coluna] = cnes[coluna].astype("string").str.strip()
cnes["regiao_cnes_lt_normalizada"] = cnes.REGSAUDE.map(
    lambda x: normaliza_codigo(x, 4)
).astype("string")

reg_hosp_raw = (cnes.dropna(subset=["regiao_cnes_lt_normalizada"])
    .groupby("CNES", as_index=False)
    .agg(regiao_saude_cnes_lt=("regiao_cnes_lt_normalizada", lambda s: s.mode().iloc[0]),
         qtd_regioes_declaradas_cnes_lt=("regiao_cnes_lt_normalizada", "nunique")))

regioes_oficiais = pd.DataFrame(
    regioes_ms_payload["macrorregiao_regiao_saude_municipios"]
).rename(columns={
    "codigo_municipio": "municipio_cod6",
    "codigo_regiao_saude": "regiao_saude",
    "regiao_saude": "regiao_saude_nome",
    "codigo_macrorregiao_saude": "macrorregiao_saude_codigo",
    "macrorregiao_saude": "macrorregiao_saude_nome",
})
for coluna in ["municipio_cod6", "regiao_saude", "macrorregiao_saude_codigo"]:
    regioes_oficiais[coluna] = regioes_oficiais[coluna].astype("string").str.strip()
regioes_oficiais = regioes_oficiais[[
    "municipio_cod6", "regiao_saude", "regiao_saude_nome",
    "macrorregiao_saude_codigo", "macrorregiao_saude_nome",
]].drop_duplicates("municipio_cod6")

cadastro_atual = pd.DataFrame(cnes_atual_payload["registros"])
cadastro_atual["CNES"] = (
    cadastro_atual.codigo_cnes.astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(7)
)
cadastro_atual = cadastro_atual.rename(columns={
    "nome_fantasia": "hospital_nome_atual",
    "nome_razao_social": "hospital_razao_social_atual",
    "descricao_esfera_administrativa": "esfera_administrativa_atual",
    "data_atualizacao": "cadastro_cnes_atualizado_em",
})

hospitais_sih = set(sih.CNES.unique())
perfil_hosp = (cnes[cnes.CNES.isin(hospitais_sih)]
    .sort_values(["CNES", "_ano_arquivo", "_mes_arquivo"])
    .groupby("CNES", as_index=False)
    .agg(municipio_cod6=("CODUFMUN", "last"), tipo_unidade_cod=("TP_UNID", "last"),
         esfera_cod_cnes_lt=("ESFERA_A", "last"), natureza_jur_cod=("NAT_JUR", "last"),
         gestao_cod=("TPGESTAO", "last")))
dim_hospital = (perfil_hosp
    .merge(reg_hosp_raw, on="CNES", how="left", validate="one_to_one")
    .merge(regioes_oficiais, on="municipio_cod6", how="left", validate="many_to_one")
    .merge(cadastro_atual[[
        "CNES", "hospital_nome_atual", "hospital_razao_social_atual",
        "esfera_administrativa_atual", "cadastro_cnes_atualizado_em",
    ]], on="CNES", how="left", validate="one_to_one"))
dim_hospital["origem_regiao"] = np.where(
    dim_hospital.regiao_saude.notna(),
    "referencia_oficial_municipio_ms",
    "sem_regiao_oficial",
)
dim_hospital["fl_regiao_conflitante"] = (
    dim_hospital.qtd_regioes_declaradas_cnes_lt.gt(1).astype("int8")
)
dim_hospital["fl_regiao_nao_confiavel"] = dim_hospital.regiao_saude.isna().astype("int8")
dim_hospital["tipo_unidade"] = dim_hospital.tipo_unidade_cod.map(DEPARA_TP_UNID)
dim_hospital["gestao"] = dim_hospital.gestao_cod.map(DEPARA_GESTAO)
dim_hospital["natureza_juridica"] = dim_hospital.natureza_jur_cod.map(DEPARA_NAT_JUR)
dim_hospital["fl_esfera_ausente_cnes_lt"] = (
    dim_hospital.esfera_cod_cnes_lt.fillna("").eq("").astype("int8")
)
dim_hospital["fl_cadastro_atual_nao_historico"] = np.int8(1)

dim_municipio = pd.DataFrame([{
    "municipio_cod7": str(item["id"]), "municipio_cod6": str(item["id"])[:6],
    "municipio_nome": item["nome"], "uf": "SP",
    "microrregiao": item["microrregiao"]["nome"],
    "mesorregiao": item["microrregiao"]["mesorregiao"]["nome"],
} for item in ibge_raw])
dim_municipio = dim_municipio.merge(
    regioes_oficiais, on="municipio_cod6", how="left", validate="one_to_one"
)
populacao_municipio = regioes_csv_sp[[
    "municipio_cod6", "qt_populacao_ibge_2022",
    "cod_regiao_de_saude", "cod_macrorregiao_de_saude",
]].rename(columns={
    "cod_regiao_de_saude": "regiao_saude_csv",
    "cod_macrorregiao_de_saude": "macrorregiao_saude_csv",
})
assert len(populacao_municipio) == 645
dim_municipio = dim_municipio.merge(
    populacao_municipio, on="municipio_cod6", how="left", validate="one_to_one"
)
assert dim_municipio.qt_populacao_ibge_2022.notna().all()
assert dim_municipio.regiao_saude.eq(dim_municipio.regiao_saude_csv).all()
assert dim_municipio.macrorregiao_saude_codigo.eq(
    dim_municipio.macrorregiao_saude_csv
).all()
dim_municipio = dim_municipio.drop(
    columns=["regiao_saude_csv", "macrorregiao_saude_csv"]
)
dim_municipio["ds_fonte_populacao"] = "Ministério da Saúde / IBGE 2022"

nat_observada = set(cnes.NAT_JUR.dropna().astype(str).unique())
inventario.append({
    "campo": "NAT_JUR", "qtd_codigos_observados": len(nat_observada),
    "codigos_sem_depara": ", ".join(sorted(nat_observada - set(DEPARA_NAT_JUR))) or "nenhum",
    "cobertura_linhas_pct": round(cnes.NAT_JUR.isin(DEPARA_NAT_JUR).mean() * 100, 6),
    "fonte": "CONCLA/IBGE 2021", "status": "mapeado",
})

print(dim_hospital.origem_regiao.value_counts(dropna=False).to_string())
print("hospitais com nome atual:", dim_hospital.hospital_nome_atual.notna().sum())
print("hospitais com esfera atual:", dim_hospital.esfera_administrativa_atual.notna().sum())
print("hospitais com natureza jurídica:", dim_hospital.natureza_juridica.notna().sum())


origem_regiao
referencia_oficial_municipio_ms    653
hospitais com nome atual: 653
hospitais com esfera atual: 653
hospitais com natureza jurídica: 653


## 3. Fatos Silver

`N_AIH`, `IDENT` e `COD_IDADE` permanecem na granularidade original.
`aih_aprovada` conta cada linha mensal; `internacao_nova` é `IDENT=1`;
`continuacao_longa_permanencia` é `IDENT=5`.

`QT_DIARIAS` significa diárias faturadas. `DIAS_PERM` representa permanência e
é o campo usado nos insumos de tempo médio. Nenhum registro com zero é excluído.

In [6]:
fato = sih.rename(columns={
    "MUNIC_MOV": "municipio_cod6", "MUNIC_RES": "municipio_res_cod6",
    "DIAG_PRINC": "cid_principal",
}).copy()
fato["dt_internacao"] = pd.to_datetime(fato.DT_INTER, format="%Y%m%d", errors="coerce")
fato["dt_saida"] = pd.to_datetime(fato.DT_SAIDA, format="%Y%m%d", errors="coerce")
fato["especialidade"] = fato.ESPEC.map(DEPARA_ESPEC)
fato["ident_descricao"] = fato.IDENT.map(DEPARA_IDENT)
fato["unidade_idade"] = fato.COD_IDADE.map(DEPARA_COD_IDADE)
fato["sexo_descricao"] = fato.SEXO.map(DEPARA_SEXO)
fato["carater_internacao"] = fato.CAR_INT.map(DEPARA_CAR_INT)
fato["complexidade"] = fato.COMPLEX.map(DEPARA_COMPLEX)
fato["marca_uti_descricao"] = fato.MARCA_UTI.map(DEPARA_UTI)

fato["idade_anos_aprox"] = np.select(
    [fato.COD_IDADE.eq("2"), fato.COD_IDADE.eq("3"), fato.COD_IDADE.eq("4"), fato.COD_IDADE.eq("5")],
    [fato.IDADE / 365.25, fato.IDADE / 12, fato.IDADE, 100 + fato.IDADE],
    default=np.nan,
)
fato["fl_aih_aprovada"] = np.int8(1)
fato["fl_internacao_nova"] = fato.IDENT.eq("1").astype("int8")
fato["fl_continuacao_longa_permanencia"] = fato.IDENT.eq("5").astype("int8")
fato["dias_perm_internacao_nova"] = fato.DIAS_PERM * fato.fl_internacao_nova
fato["qt_diarias_internacao_nova"] = fato.QT_DIARIAS * fato.fl_internacao_nova
fato["valor_internacao_nova"] = fato.VAL_TOT * fato.fl_internacao_nova
fato["valor_continuacao"] = fato.VAL_TOT * fato.fl_continuacao_longa_permanencia
fato["fl_sem_diaria_faturada"] = fato.QT_DIARIAS.eq(0).astype("int8")
fato["fl_permanencia_zero"] = fato.DIAS_PERM.eq(0).astype("int8")
fato["fl_sem_valor"] = fato.VAL_TOT.eq(0).astype("int8")
fato["fl_obito"] = fato.MORTE.eq(1).astype("int8")
fato["fl_obito_internacao_nova"] = (fato.MORTE.eq(1) & fato.IDENT.eq("1")).astype("int8")
fato["fl_aih_com_valor"] = fato.VAL_TOT.gt(0).astype("int8")
fato["fl_uti"] = (fato.MARCA_UTI.ne("00") | fato.UTI_MES_TO.gt(0)).astype("int8")
fato["fl_cruza_mes"] = (
    fato.dt_internacao.notna() & fato.dt_saida.notna()
    & fato.dt_internacao.dt.to_period("M").ne(fato.dt_saida.dt.to_period("M"))
).astype("int8")
fato["fl_competencia_diverge_saida"] = (
    fato.dt_saida.notna()
    & ((fato.dt_saida.dt.year != fato._ano) | (fato.dt_saida.dt.month != fato._mes))
).astype("int8")

fato = (fato.merge(
    dim_hospital[["CNES", "regiao_saude", "regiao_saude_nome",
                  "macrorregiao_saude_codigo", "macrorregiao_saude_nome",
                  "origem_regiao", "fl_regiao_conflitante",
                  "fl_regiao_nao_confiavel"]],
    on="CNES", how="left", validate="many_to_one")
    .merge(dim_cid[["cid_principal", "cid_descricao", "categoria_descricao",
                    "capitulo", "capitulo_desc", "fonte_descricao"]],
           on="cid_principal", how="left", validate="many_to_one"))

COLS_FATO = [
    "N_AIH", "IDENT", "ident_descricao", "CNES", "municipio_cod6", "municipio_res_cod6",
    "regiao_saude", "regiao_saude_nome", "macrorregiao_saude_codigo",
    "macrorregiao_saude_nome", "origem_regiao", "fl_regiao_conflitante",
    "fl_regiao_nao_confiavel",
    "_ano", "_mes", "dt_internacao", "dt_saida", "fl_cruza_mes",
    "fl_competencia_diverge_saida", "ESPEC", "especialidade", "cid_principal",
    "cid_descricao", "categoria_descricao", "capitulo", "capitulo_desc",
    "fonte_descricao", "QT_DIARIAS", "DIAS_PERM", "MORTE", "VAL_TOT",
    "UTI_MES_TO", "MARCA_UTI", "marca_uti_descricao", "IDADE", "COD_IDADE",
    "unidade_idade", "idade_anos_aprox", "SEXO", "sexo_descricao", "CAR_INT",
    "carater_internacao", "COMPLEX", "complexidade", "fl_aih_aprovada",
    "fl_internacao_nova", "fl_continuacao_longa_permanencia",
    "dias_perm_internacao_nova", "qt_diarias_internacao_nova",
    "valor_internacao_nova", "valor_continuacao",
    "fl_sem_diaria_faturada", "fl_permanencia_zero", "fl_sem_valor",
    "fl_obito", "fl_obito_internacao_nova", "fl_aih_com_valor", "fl_uti",
    "_arquivo_fonte",
]
fato_internacao = fato[COLS_FATO]
print("fato_internacao:", fato_internacao.shape)

fato_internacao: (7034961, 59)


In [7]:
for coluna in ["CNES", "CODLEITO"]:
    cnes[coluna] = cnes[coluna].astype("string").str.strip()
for coluna in ["QT_SUS", "QT_EXIST"]:
    cnes[coluna] = pd.to_numeric(cnes[coluna], errors="raise").astype("int32")
cnes["_ano"] = cnes["_ano_arquivo"].astype("int16")
cnes["_mes"] = cnes["_mes_arquivo"].astype("int8")

fato_leitos_mensal = (cnes.groupby(["CNES", "_ano", "_mes"], as_index=False, dropna=False)
    .agg(leitos_sus=("QT_SUS", "sum"), leitos_totais=("QT_EXIST", "sum"),
         tipos_de_leito=("CODLEITO", "nunique")))
fato_leitos_mensal = fato_leitos_mensal[fato_leitos_mensal.CNES.isin(hospitais_sih)].copy()
fato_leitos_mensal = fato_leitos_mensal.merge(
    dim_tempo[["_ano", "_mes", "dias_no_mes"]], on=["_ano", "_mes"], how="left", validate="many_to_one")
fato_leitos_mensal["capacidade_teorica_leito_dia"] = (
    fato_leitos_mensal.leitos_sus * fato_leitos_mensal.dias_no_mes
)

## 4. Bases agregadas sem perda de nulos

Todos os `groupby` analíticos usam `dropna=False`. Cada soma é reconciliada
contra o fato antes da gravação. A base CID usa somente internações novas e
`DIAS_PERM`; região ausente permanece visível.

In [8]:
def agregar_base(frame, chaves):
    return (frame.groupby(chaves, as_index=False, dropna=False)
        .agg(aih_aprovadas=("fl_aih_aprovada", "sum"),
             aih_distintas=("N_AIH", "nunique"),
             internacoes_novas=("fl_internacao_nova", "sum"),
             continuacoes_longa_permanencia=("fl_continuacao_longa_permanencia", "sum"),
             qt_diarias_soma=("QT_DIARIAS", "sum"),
             qt_diarias_internacoes_novas_soma=("qt_diarias_internacao_nova", "sum"),
             dias_perm_soma=("DIAS_PERM", "sum"),
             dias_perm_internacoes_novas_soma=("dias_perm_internacao_nova", "sum"),
             obitos_aih=("fl_obito", "sum"),
             obitos_internacoes_novas=("fl_obito_internacao_nova", "sum"),
             valor_total=("VAL_TOT", "sum"),
             valor_internacoes_novas=("valor_internacao_nova", "sum"),
             valor_continuacoes=("valor_continuacao", "sum"),
             aih_com_valor=("fl_aih_com_valor", "sum"),
             registros_uti=("fl_uti", "sum")))

base_hospital_mes = agregar_base(fato_internacao, ["CNES", "_ano", "_mes"])
base_hospital_mes = (base_hospital_mes
    .merge(fato_leitos_mensal[["CNES", "_ano", "_mes", "leitos_sus", "dias_no_mes",
                               "capacidade_teorica_leito_dia"]],
           on=["CNES", "_ano", "_mes"], how="left", validate="one_to_one")
    .merge(dim_hospital[["CNES", "municipio_cod6", "hospital_nome_atual",
                         "regiao_saude", "regiao_saude_nome",
                         "macrorregiao_saude_codigo", "macrorregiao_saude_nome",
                         "origem_regiao", "fl_regiao_nao_confiavel"]],
           on="CNES", how="left", validate="many_to_one"))
base_hospital_mes["permanencia_media_internacoes_novas"] = np.where(
    base_hospital_mes.internacoes_novas > 0,
    base_hospital_mes.dias_perm_internacoes_novas_soma / base_hospital_mes.internacoes_novas,
    np.nan,
)
base_hospital_mes["proxy_iph_diarias_faturadas"] = np.where(
    base_hospital_mes.capacidade_teorica_leito_dia > 0,
    base_hospital_mes.qt_diarias_soma / base_hospital_mes.capacidade_teorica_leito_dia,
    np.nan,
)
base_hospital_mes["status_proxy_iph"] = "experimental_nao_validado_como_ocupacao_real"

base_hospital_espec_mes = agregar_base(
    fato_internacao, ["CNES", "ESPEC", "especialidade", "_ano", "_mes"])
base_hospital_espec_mes = base_hospital_espec_mes.merge(
    dim_hospital[["CNES", "municipio_cod6", "hospital_nome_atual",
                  "regiao_saude", "regiao_saude_nome", "origem_regiao"]],
    on="CNES", how="left", validate="many_to_one")

internacoes_novas = fato_internacao[fato_internacao.fl_internacao_nova.eq(1)]
base_hospital_cid = (internacoes_novas.groupby(
    ["CNES", "regiao_saude", "origem_regiao", "cid_principal", "capitulo"],
    as_index=False, dropna=False)
    .agg(internacoes_novas=("fl_internacao_nova", "sum"),
         dias_perm_soma=("DIAS_PERM", "sum"),
         qt_diarias_soma=("QT_DIARIAS", "sum"),
         obitos=("fl_obito", "sum"), valor_total=("VAL_TOT", "sum")))
base_hospital_cid["permanencia_media"] = (
    base_hospital_cid.dias_perm_soma / base_hospital_cid.internacoes_novas
)

## 5. Reconciliação e situação dos indicadores

- **TMH e CMI:** óbito, valor e internação nova são reconciliados aqui; as
  fórmulas aprovadas pertencem à Gold.
- **IPR:** `DIAS_PERM` é preservado sem excluir zero e sem perder região nula.
- **IS:** competência e unidade de contagem ficam explícitas para a série Gold.
- **IPH:** `QT_DIARIAS` permanece apenas para auditoria do proxy antigo. A Gold
  reconstrói pacientes-dia pelas datas e não chama o resultado de ocupação real.

In [9]:
esperado_sih = manifesto["checks"]["linhas_sih"]
esperado_cnes = manifesto["checks"]["linhas_cnes"]
metricas = {
    "linhas_sih_reconciliadas": len(fato_internacao),
    "linhas_cnes_reconciliadas": len(cnes),
    "aih_aprovadas": int(fato_internacao.fl_aih_aprovada.sum()),
    "aih_distintas": int(fato_internacao.N_AIH.nunique()),
    "internacoes_novas": int(fato_internacao.fl_internacao_nova.sum()),
    "continuacoes_longa_permanencia": int(fato_internacao.fl_continuacao_longa_permanencia.sum()),
    "especialidades_sem_depara": int(fato_internacao.especialidade.isna().sum()),
    "hospitais_sem_match_cnes": len(set(sih.CNES) - set(dim_hospital.CNES)),
    "cids_sem_capitulo": int(fato_internacao.capitulo.eq("--").sum()),
    "cids_sem_descricao": int(fato_internacao.cid_descricao.isna().sum()),
    "hospitais_sem_nome_atual": int(dim_hospital.hospital_nome_atual.isna().sum()),
    "hospitais_sem_esfera_atual": int(dim_hospital.esfera_administrativa_atual.isna().sum()),
    "hospitais_sem_natureza_juridica": int(dim_hospital.natureza_juridica.isna().sum()),
    "registros_sem_regiao": int(fato_internacao.regiao_saude.isna().sum()),
    "internacoes_novas_sem_regiao": int(
        (fato_internacao.fl_internacao_nova.eq(1) & fato_internacao.regiao_saude.isna()).sum()),
    "hospitais_regiao_conflitante": int(dim_hospital.fl_regiao_conflitante.sum()),
    "qt_diarias_igual_dias_perm_pct": round(
        fato_internacao.QT_DIARIAS.eq(fato_internacao.DIAS_PERM).mean() * 100, 4),
    "qt_zero_dias_perm_positivo": int(
        (fato_internacao.QT_DIARIAS.eq(0) & fato_internacao.DIAS_PERM.gt(0)).sum()),
    "cruza_mes_pct": round(fato_internacao.fl_cruza_mes.mean() * 100, 4),
    "competencia_diverge_saida_pct": round(
        fato_internacao.fl_competencia_diverge_saida.mean() * 100, 4),
    "tmh_internacoes_novas_pct": round(
        fato_internacao.fl_obito_internacao_nova.sum()
        / fato_internacao.fl_internacao_nova.sum() * 100, 4),
    "proxy_iph_media_hospital_mes": round(base_hospital_mes.proxy_iph_diarias_faturadas.mean(), 6),
}

assert len(fato_internacao) == esperado_sih
assert len(cnes) == esperado_cnes
assert metricas["aih_aprovadas"] == esperado_sih
assert (
    metricas["internacoes_novas"] + metricas["continuacoes_longa_permanencia"]
    == esperado_sih
)
assert metricas["especialidades_sem_depara"] == 0
assert metricas["cids_sem_capitulo"] == 0
assert metricas["cids_sem_descricao"] == 0
assert metricas["hospitais_sem_nome_atual"] == 0
assert metricas["hospitais_sem_esfera_atual"] == 0
assert metricas["hospitais_sem_natureza_juridica"] == 0
assert metricas["registros_sem_regiao"] == 0
assert set(fato_internacao.CNES) <= set(dim_hospital.CNES)
assert base_hospital_mes.aih_aprovadas.sum() == esperado_sih
assert base_hospital_espec_mes.aih_aprovadas.sum() == esperado_sih
assert base_hospital_mes.dias_perm_internacoes_novas_soma.sum() == internacoes_novas.DIAS_PERM.sum()
assert base_hospital_espec_mes.dias_perm_internacoes_novas_soma.sum() == internacoes_novas.DIAS_PERM.sum()
assert np.isclose(
    base_hospital_mes.valor_total.sum(),
    base_hospital_mes.valor_internacoes_novas.sum() + base_hospital_mes.valor_continuacoes.sum(),
)
assert base_hospital_cid.internacoes_novas.sum() == metricas["internacoes_novas"]
assert (
    base_hospital_cid.loc[base_hospital_cid.regiao_saude.isna(), "internacoes_novas"].sum()
    == internacoes_novas.regiao_saude.isna().sum()
), "groupby perdeu internações com região nula"

status_indices = pd.DataFrame([
    {"indice": "TMH", "status": "contrato_aprovado",
     "regra": "óbitos / internações novas; mínimo de 30 para classificação"},
    {"indice": "IPR", "status": "contrato_aprovado",
     "regra": "permanência hospital/CID / benchmark regional sem o hospital"},
    {"indice": "IS", "status": "contrato_aprovado",
     "regra": "2026 / média do mesmo mês em 2024 e 2025"},
    {"indice": "CMI", "status": "contrato_aprovado",
     "regra": "valor aprovado / internações novas; continuações separadas"},
    {"indice": "IPH", "status": "contrato_aprovado_com_limitacao",
     "regra": "pacientes-dia estimados / leitos-dia declarados; não é ocupação real"},
])
print(pd.Series(metricas).to_string())
print("\n", status_indices.to_string(index=False))

linhas_sih_reconciliadas           7.034961e+06
linhas_cnes_reconciliadas          2.430850e+05
aih_aprovadas                      7.034961e+06
aih_distintas                      6.909807e+06
internacoes_novas                  6.905441e+06
continuacoes_longa_permanencia     1.295200e+05
especialidades_sem_depara          0.000000e+00
hospitais_sem_match_cnes           0.000000e+00
cids_sem_capitulo                  0.000000e+00
cids_sem_descricao                 0.000000e+00
hospitais_sem_nome_atual           0.000000e+00
hospitais_sem_esfera_atual         0.000000e+00
hospitais_sem_natureza_juridica    0.000000e+00
registros_sem_regiao               0.000000e+00
internacoes_novas_sem_regiao       0.000000e+00
hospitais_regiao_conflitante       4.000000e+00
qt_diarias_igual_dias_perm_pct     7.005470e+01
qt_zero_dias_perm_positivo         1.815840e+05
cruza_mes_pct                      1.507980e+01
competencia_diverge_saida_pct      1.876330e+01
tmh_internacoes_novas_pct          5.107

## 6. Gravação e documentação da Silver

As bases só são promovidas após todas as reconciliações acima passarem.

In [10]:
SAIDAS_SILVER = {
    "dim_tempo": dim_tempo,
    "dim_hospital": dim_hospital,
    "dim_municipio": dim_municipio,
    "dim_especialidade": dim_especialidade,
    "dim_cid": dim_cid,
    "dim_dominio": dim_dominio,
    "fato_internacao": fato_internacao,
    "fato_leitos_mensal": fato_leitos_mensal,
}
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
from pipeline.contratos import publicar_silver

saidas_canonicas = publicar_silver(
    base=BASE,
    saidas_originais=SAIDAS_SILVER,
    inventario=inventario,
    metricas=metricas,
    status_indices=status_indices,
    manifesto_bronze=manifesto,
    sobrescrever=SOBRESCREVER,
)
print("\nSILVER VÁLIDA — saídas e documentação gravadas.")


dim_tempo                            29 linhas
dim_hospital                        653 linhas
dim_municipio                       645 linhas
dim_especialidade                    15 linhas
dim_cid                           9,494 linhas
dim_dominio                          84 linhas
fato_internacao               7,034,961 linhas
fato_leito_mensal                18,690 linhas



SILVER VÁLIDA — saídas e documentação gravadas.
